# 第 12 讲：图论与网络模型

> 用 networkx 落成最短路、最大流和网络中心性示例。

## 使用说明

- 先从上到下运行全部单元。
- 每讲都会生成一个 `outputs/lesson-xx/` 目录，用于保存图表、表格或报告片段。
- 示例数据均为教学用合成数据，真实比赛中应替换为题目附件数据。

In [ ]:
LESSON_ID = "lesson-12"

from pathlib import Path
import os
import math
import itertools
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

OUTPUT_ROOT = Path(os.environ.get("COURSE_OUTPUT_DIR", REPO_ROOT / "outputs")) / LESSON_ID
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(42)
print(f"Output directory: {OUTPUT_ROOT}")

In [ ]:
import networkx as nx

edges = [
    ("A", "B", 4), ("A", "C", 2), ("B", "D", 5), ("C", "D", 8),
    ("C", "E", 10), ("D", "E", 2), ("D", "F", 6), ("E", "F", 3),
]
G = nx.Graph()
G.add_weighted_edges_from(edges)
path = nx.shortest_path(G, "A", "F", weight="weight")
dist = nx.shortest_path_length(G, "A", "F", weight="weight")
centrality = pd.Series(nx.betweenness_centrality(G, weight="weight"), name="betweenness")
centrality.to_csv(OUTPUT_ROOT / "node_centrality.csv", encoding="utf-8-sig")
print("Shortest path:", path, "distance:", dist)
centrality.sort_values(ascending=False)

In [ ]:
pos = nx.spring_layout(G, seed=42)
fig, ax = plt.subplots(figsize=(7, 5))
nx.draw_networkx(G, pos=pos, ax=ax, node_color="lightblue", node_size=900)
labels = nx.get_edge_attributes(G, "weight")
nx.draw_networkx_edge_labels(G, pos, edge_labels=labels, ax=ax)
ax.set_title("Weighted network")
ax.axis("off")
fig.tight_layout()
fig.savefig(OUTPUT_ROOT / "weighted_network.png", dpi=160)
plt.show()

In [ ]:
DG = nx.DiGraph()
DG.add_edge("S", "A", capacity=10)
DG.add_edge("S", "B", capacity=8)
DG.add_edge("A", "C", capacity=5)
DG.add_edge("A", "D", capacity=6)
DG.add_edge("B", "C", capacity=4)
DG.add_edge("B", "D", capacity=4)
DG.add_edge("C", "T", capacity=8)
DG.add_edge("D", "T", capacity=9)
flow_value, flow_dict = nx.maximum_flow(DG, "S", "T")
pd.DataFrame(flow_dict).fillna(0).to_csv(OUTPUT_ROOT / "max_flow.csv", encoding="utf-8-sig")
flow_value, flow_dict